In [10]:
import argparse
from logging import getLogger

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import init_logger, init_seed, set_color
from recbole.model.general_recommender import NCL
from recbole.trainer import NCLTrainer

#from models.lightgcn import LightGCN
#from models.ncl import NCL
from models.scl import SCL
#from trainers.lightgcn_trainer import LightGCNTrainer
#from trainers.ncl_trainer import NCLTrainer

import time
import json
import os

In [11]:
config = Config(
    model='NCL',
    dataset='travel',
    config_file_list=['configs/overall.yaml', 'configs/ncl.yaml', 'configs/travel.yaml'])

init_seed(2020, True)

# logger initialization
init_logger(config)
logger = getLogger()
logger.info(config)

# dataset filtering
dataset = create_dataset(config)
logger.info(dataset)

# dataset splitting
train_data, valid_data, test_data = data_preparation(config, dataset)

16 Mar 12:50    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = ./travel
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 1000
train_batch_size = 4096
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 4

Evaluation Hyper Parameters:
eval_args = {'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}
repeatable = False
metrics = ['Recall', 'NDCG', 'Precision', 'MAP']
topk = [1, 2, 3, 4, 5, 10, 20, 30, 50]
valid_metric = NDCG@10
valid_metric_bigger = True
eval_batch_size = 4096
metric_decimal_place = 4

Dataset Hype

In [12]:
train_data.dataset

travel
The number of users: 2010
Average actions of users: 15.115978098556496
The number of items: 247
Average actions of items: 123.44715447154472
The number of inters: 30368
The sparsity of the dataset: 93.88321550144016%
Remain Fields: ['user_id', 'item_id', 'rating', 'post_num', 'follower', 'following', 'timestamp', 'neg_item_id']

In [13]:
valid_data.dataset

travel
The number of users: 2010
Average actions of users: 2.1168057210965436
The number of items: 247
Average actions of items: 15.114893617021277
The number of inters: 3552
The sparsity of the dataset: 99.28454891534231%
Remain Fields: ['user_id', 'item_id', 'rating', 'post_num', 'follower', 'following', 'timestamp', 'neg_item_id']

In [14]:
test_data.dataset

travel
The number of users: 2010
Average actions of users: 2.03137039075399
The number of items: 247
Average actions of items: 15.90948275862069
The number of inters: 3691
The sparsity of the dataset: 99.25655125183798%
Remain Fields: ['user_id', 'item_id', 'rating', 'post_num', 'follower', 'following', 'timestamp', 'neg_item_id']

In [15]:
model = NCL(config, train_data.dataset).to(config['device'])
trainer = NCLTrainer(config, model)

In [16]:
# model training
start_time = time.time()

best_valid_score, best_valid_result = trainer.fit(
    train_data, valid_data, saved=True, show_progress=False
)

end_time = time.time()
ncl_time = end_time - start_time

16 Mar 12:50    INFO  Running E-step ! 
/data1/heejung/envs/travel/lib/python3.10/site-packages/recbole/trainer/trainer.py:1440: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)
/data1/heejung/envs/travel/lib/python3.10/site-packages/recbole/trainer/trainer.py:1453: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(enabled=self.enable_amp):
16 Mar 12:50    INFO  epoch 0 training [time: 0.17s, train_loss1: 5.5430, train_loss2: 0.0196]
16 Mar 12:50    INFO  epoch 0 evaluating [time: 0.11s, valid_score: 0.028900]
16 Mar 12:50    INFO  valid result: 
recall@1 : 0.0054    recall@2 : 0.0109    recall@3 : 0.0209    recall@4 : 0.0251    recall@5 : 0.0275    recall@10 : 0.0489    recall@20 : 0.1102    recall@30 : 0.1567    recall@50 : 0.2445    ndcg@1 : 0.0143    ndcg@2 : 0

In [17]:
# model evaluation
NCL_result = trainer.evaluate(test_data, load_best_model=False, show_progress=False)

logger.info(set_color('best valid ', 'yellow') + f': {best_valid_result}')
logger.info(set_color('test result', 'yellow') + f': {NCL_result}')

16 Mar 12:51    INFO  best valid : OrderedDict([('recall@1', 0.1424), ('recall@2', 0.2167), ('recall@3', 0.2828), ('recall@4', 0.3282), ('recall@5', 0.3645), ('recall@10', 0.5101), ('recall@20', 0.6431), ('recall@30', 0.7179), ('recall@50', 0.8088), ('ndcg@1', 0.23), ('ndcg@2', 0.2444), ('ndcg@3', 0.2677), ('ndcg@4', 0.2835), ('ndcg@5', 0.2962), ('ndcg@10', 0.3482), ('ndcg@20', 0.388), ('ndcg@30', 0.4082), ('ndcg@50', 0.4298), ('precision@1', 0.23), ('precision@2', 0.1815), ('precision@3', 0.1627), ('precision@4', 0.1457), ('precision@5', 0.1313), ('precision@10', 0.0967), ('precision@20', 0.0637), ('precision@30', 0.049), ('precision@50', 0.0341), ('map@1', 0.23), ('map@2', 0.2177), ('map@3', 0.2247), ('map@4', 0.2311), ('map@5', 0.2366), ('map@10', 0.2598), ('map@20', 0.2736), ('map@30', 0.2796), ('map@50', 0.2848)])
16 Mar 12:51    INFO  test result: OrderedDict([('recall@1', 0.1741), ('recall@2', 0.2616), ('recall@3', 0.3187), ('recall@4', 0.3604), ('recall@5', 0.3978), ('recall@10